In [7]:
from minesweep import Board
from show import show_board
from constants import *

import torch
import torch.nn as nn
import numpy as np

In [2]:
#Generate dataset

def generate_dataset(n: int, w: int = 5, h: int = 5) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Xs: tensor stackejat de N tensors 5x5
    ys: tensor amb N caselles del mig
        True if bomb False else
    """
    Xs = []
    ys = []
    for _ in range(n):
        b = Board(w, h)
        found_bomb = b.get_board()[w//2][h//2] == -1
        b.hide_cell(w//2, h//2)

        Xs.append(torch.as_tensor(b.get_board(), dtype=torch.int8))
        ys.append(found_bomb)
        # torch.tensor(ys)

    return torch.stack(Xs), torch.tensor(ys)    # shape (n, w, h)


In [3]:

DATASET_SIZE = 1000
data = generate_dataset(DATASET_SIZE, 5, 5)

print(data[0].shape, data[0].dtype, data[1].shape, data[1].dtype) 

torch.save(data, "boards_5x5.pt")

torch.Size([1000, 5, 5]) torch.int8 torch.Size([1000]) torch.bool


In [8]:
class MinesweeperNN(nn.Module):

    xarxa:nn.Sequential
    def processa_tauler(b:np.ndarray) -> torch.Tensor:
        w, h = board.shape
        out = np.zeros((NUM_CHANNELS, 5, 5), dtype=np.float32) #12 canals
    
        for i in range(w):
            for j in range(h):
                match cella:
                    case constants.BOMB_CELL:
                        raise ValueError("Algo no ha anat bé, la NN no pot rebre bombes")

                    case constants.OUT_OF_BOUNDS_CELL:
                        out[OUT_OF_BOUNDS_CHANNEL, i, j] = 1

                    case constants.HIDDEN_CELL:
                        out[HIDDEN_CHANNEL, i, j] = 1

                    case constants.FLAGGED_CELL:
                        out[FLAGGED_CHANNEL, i, j] = 1

                    case constants.EMPTY_CELL:
                        out[REVEALED_BASE_CHANNEL, i, j] = 1

                    case a if a < 9:
                        out[REVEALED_BASE_CHANNEL + 1, i, j] = 1
                    case _:
                        raise ValueError("what te fuuuk")
        return out


    def __init__(self, w, h):
        super().__init__() 
        self.xarxa = nn.Sequential(
            nn.Linear(w*h*NUM_CHANNELS, 20),
            nn.ReLU(),

            nn.Linear(20, 2), #output
            nn.Softmax(2)
        )

    def forward(self, tauler:Board):
        embedded_item = self.processa_tauler(tauler)
        probabilities = self.xarxa(embedded_item)

        return probabilities